# 🗺️ **Rasterization/Vectorization**

> ### 📌 **Overview**
>
> This notebook compares conversions between vector and raster formats across `rasterio`, `rioxarray`, `odc-geo`, `geoutils`, and `xarray-spatial`.
>
> **What we cover :**
> - **Rasterization** to burn vector geometries onto a raster grid
> - **Vectorization** to extract vector polygons from raster regions

In [ ]:
import numpy as np

import geopandas as gpd

import rasterio
from rasterio import features
from rasterio.plot import show
from rasterio.transform import from_origin

import rioxarray as rxr

import geoutils as gu

from xrspatial.rasterize import rasterize as xrs_rasterize
from xrspatial.polygonize import polygonize as xrs_polygonize

import matplotlib.pyplot as plt

import seaborn as sns #cmap mako

#to be used with rioxarray
from geocube.api.core import make_geocube
from geocube.vector import vectorize

from shapely.geometry import shape

In [ ]:
#path to the raster object
raster_path = "../data/rasters/subset_med.tif"

In [ ]:
#path to the vector object
vector_path = "../data/vectors/buildings.shp"

In [ ]:
rast = gu.Raster(raster_path)
vect = gu.Vector(vector_path)

rast.plot(bands=1, cmap="mako")
vect.plot(fc="green", ec="k", lw=1)

## **Rasterize a vector**

Rasterizing is available in `rasterio`, `rioxarray`, `geoutils` and `xarray-spatial` libraries, but not implemented in `odc-geo`.

This method consists of converting a vector (shape) into a raster image.

> `dask` note: built-in rasterization is generally eager; to keep scalability in a `dask` workflow, consider tiling or reducing active window size, then compute outputs.

In this example, we will transform building polygons (.shp) located in our image, to rasters.

### • *rasterio*

First, we have to create our `geopandas.DataFrame` containing geometries of our buildings polygons:

In [ ]:
gdf = gpd.read_file(vector_path)

In [ ]:
gdf["value"] = 1 #needed to use .rasterize later

Then we need to define the parameters of our output raster - transform, resolution, shape and CRS : 

In [ ]:
resolution = 0.31 #res of our image
xmin, ymin, xmax, ymax = gdf.total_bounds
transform = from_origin(xmin, ymax, resolution, resolution)

width = int((xmax - xmin) / resolution)
height = int((ymax - ymin) / resolution)

We create an iterator of (geometry, value) that we will have to pass as an argument when rasterizing :

In [ ]:
shapes = ((geom, value) for geom, value in zip(gdf["geometry"], gdf["value"]))

Finally, we will use the [`rasterize()`](https://rasterio.readthedocs.io/en/latest/api/rasterio.features.html#rasterio.features.rasterize) function:

For that, we have to pass several arguments, like the shapes we want to convert to raster, the output shape wanted (`out_shape`) and the transform matrix.

It will return an image with input geometries converted to rasters.

In [ ]:
raster = features.rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0, #value for nodata
    dtype="int32"
)

We can plot our image to see the result:

In [ ]:
show(raster, cmap="PuBuGn")

### • *rioxarray*

Rasterizing a vector is not possible directly through `rioxarray`, so we will have to use another library, namely `geocube`, and especially the [`geocube.make_geocube()`](https://corteva.github.io/geocube/html/geocube.html) function for this matter.

It will rasterize a vector into an `xarray` object. We will pass as arguments our geodataframe containing vectors, and specify the resolution of our pixels:

In [ ]:
cube = make_geocube(vector_data=gdf, resolution=(-0.31, 0.31))

The object created is an `xarray.Dataset`, as we can see:

In [ ]:
cube

We can check the result by plotting the new raster:

In [ ]:
cube["value"].plot(cmap="PuBuGn", vmin=0)
plt.show()

### • *odc-geo*

`odc-geo` provides `odc.geo.xr.rasterize(geom, how=geobox)` for geometries, though `rasterio.features.rasterize` and `geocube` are more commonly used in standard `rioxarray` workflows.

### • *geoutils*

With `geoutils`, we will use the [`rasterize()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Vector.rasterize.html#geoutils.Vector.rasterize) function:

In [ ]:
gu_rast = gu.Raster(raster_path)

gu_vect = gu.Vector(vector_path)

If we plot our polygons above the raster:

In [ ]:
gu_rast.plot(bands=1, cmap="mako")
gu_vect.plot(ref_crs=gu_rast, fc= "green", ec="k", lw=1)

Next, we use the `.rasterize()` function on our vectors to convert them to rasters. As arguments, we will pass our reference raster to match, and the values to be burned inside the polygons (`in_value`) :

In [ ]:
vect_rasterized = gu_vect.rasterize(gu_rast, in_value=1) #in_value = 1 to burn a single value

If we plot the map, here is the result, focused on our area of study:

In [ ]:
vect_rasterized.plot(cmap="PuBuGn")

### • *xarray-spatial*

Using `xarray-spatial`, we can simply use the [`.rasterize()`](https://xarray-spatial.readthedocs.io/en/stable/reference/_autosummary/xrspatial.rasterize.rasterize.html) function to rasterize vector geometries into a raster.

We will reuse the same object `GeoDataFrame` *gdf* created previously to perform the rasterization.

`xarray-spatial` also provides an interesting feature with the argument `like`, which allows to copy the *width*, *height*, *bounds*, and *dtype* from another array taken as a template. For now, we will just create a new one without any reference.

In [ ]:
gdf.head()

We then need to specify the resolution we want to have for the newly created raster. As for previous examples, we will keep it as *0.31* :

In [ ]:
xrs_rasterized = xrs_rasterize(gdf, resolution=0.31, column="value")

Finally we can plot the result :

In [ ]:
show(xrs_rasterized, cmap="PuBuGn", vmin=0)

## **Vectorize a raster**

Vectorizing a raster is available in `rasterio`, `rioxarray`, `geoutils` and `xarray-spatial` libraries, but not implemented in `odc-geo`.

This method is the opposite of the previous section, it consists of converting a raster into a vector (shape).

This time, we will focus on a raster of the north of Sardinia (small part for processing purposes), and will convert it to a shape.

In [ ]:
#path to the raster object
raster_path = "../data/rasters/sard2.tif"

### • *rasterio*

In `rasterio`, we first need to select a band and get its positional metadata (transform and CRS).

We will also create a shape object which is an iterator containing (geometry, value) pairs:

In [ ]:
with rasterio.open(raster_path) as src:
    b1 = src.read(1)
    transform = src.transform
    crs = src.crs

shapes = features.shapes(b1, transform=transform)

We will then create two lists containing our geometries (converted from `geojson` to `shapely`) and the associated values for each geometry:

In [ ]:
geom_list = []
value_list = []

for geom, value in shapes:
    geom_list.append(shape(geom)) #convert from geojson to shapely
    value_list.append(value)

After that, we have to create our `GeoDataFrame`. We will pass as arguments our lists previously created, and the CRS:

In [ ]:
gdf = gpd.GeoDataFrame({"value": value_list, "geometry": geom_list}, crs=crs)

Finally, we can plot our new vector:

In [ ]:
gdf.plot(column="value", cmap="PuBuGn")
plt.show()

### • *rioxarray*

For `rioxarray`, we will use again the geocube library and especially the [`geocube.vectorize()`](https://corteva.github.io/geocube/html/geocube.html#geocube.vector.vectorize) function:

In [ ]:
da_rxr = rxr.open_rasterio(raster_path)

We now use the `vectorize()` function from `geocube`. We just need to pass the raster to be vectorized as an argument:

In [ ]:
da_rxr_vectorized = vectorize(da_rxr)

We can finally plot the result:

In [ ]:
da_rxr_vectorized.plot(cmap="mako")

### • *odc-geo*

`odc-geo` does not provide a built-in raster-to-vector polygonization function. We can use `rasterio.features.shapes()` or `geoutils.Raster.polygonize()` instead.

### • *geoutils*

With `geoutils`, we will use the [`polygonize()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.polygonize.html#geoutils.Raster.polygonize) function to vectorize our raster:

In [ ]:
gu_rast = gu.Raster(raster_path)

Here is the raster before being converted to a vector:

In [ ]:
gu_rast.plot(cmap="mako")

Here we call the function `polygonize`:

In [ ]:
rast_polygonized = gu_rast.polygonize()

The returned object is a `geoutils.Vector` containing all vectorized geometries and their values:

In [ ]:
rast_polygonized

And we can finally plot the result:

In [ ]:
rast_polygonized.plot(cmap="mako")

### • *xarray-spatial*

`xarray-spatial` also has its own function to vectorize rasters, called [`.polygonize()`](https://xarray-spatial.readthedocs.io/en/stable/reference/_autosummary/xrspatial.polygonize.polygonize.html).

The `.polygonize()` function is compatible with `dask` :
> polygonize runs per chunk and stitches the results, so it never holds the whole raster in memory at once. Chunks are polygonized in batches: each batch is one dask.compute call, so dask schedules the batch in parallel instead of computing one chunk at a time. Peak memory is bounded by one batch worth of per-chunk polygons plus the boundary polygons that accumulate for the merge. Larger inputs with many small chunks parallelize well; rasters whose chunks each produce many polygons use more memory per batch.

Once again, we will take the same vector (north of Sardinia) to perform the vectorization.

We can set the *return_type* argument as `geopandas` to return a `GeoDataFrame` after the vectorization, instead of numpy values by default.

First, we will select only the first band, to provide a 2D DataArray as the input :

In [ ]:
da_rxr_b1 = da_rxr.isel(band=0)

In [ ]:
xrs_polygonized = xrs_polygonize(da_rxr_b1, return_type="geopandas")

We can finally plot the results :

In [ ]:
xrs_polygonized.plot(cmap="mako")